Processar o arquivo de base com as perguntas.

In [9]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-06 21:01:33--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8498 (8.3K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]   8.30K  --.-KB/s    in 0s      

2026-04-06 21:01:33 (62.8 MB/s) - ‘questions.ipynb’ saved [8498/8498]



Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [10]:
# Instalar ou atualizar as bibliotecas necessárias.
!pip install -q -U openai bert-score

# Importar bibliotecas.
import pandas as pd
import os
from openai import OpenAI
from google import genai
from google.colab import userdata
from bert_score import score

Funcões dos clientes das IAs, modelos e realização de consultas.

In [11]:
# Iniciar o cliente da IA
def client_ai_instance(name_api_key):
  # Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
  # O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
  # Tal chave previamente criada no Google AI Studio ou no Console do Groq.
  # Observação: o nome da chave definido precisa ser o mesmo inclusive com diferenciação de letra maiúscula e minúscula.
  api_key = userdata.get(name_api_key)

  # Verificar se foi o groq ou google.
  if name_api_key == 'groq_api_key':
    client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=api_key
  )
  elif name_api_key == 'oprouter_api_key':
    client_ai = OpenAI(
      base_url="https://openrouter.ai/api/v1/",
      api_key=api_key
    )
  elif name_api_key == 'google_api_key':
    client_ai = genai.Client(api_key=api_key)
  else:
    print('chave não encontrada')

  # Retornar instancia do cliente da IA.
  return client_ai

  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )
  return client_ai

def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

def questions_submit(client_ai, model_id, df_questions):
    # Criar lista vazia para guardar as respostas.
    responses = []

    # Repetição para percorrer todo Dataframe.
    for index, row in df_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preencher prompt do usuário.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Realizar consulta a IA.
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          max_tokens=1024,
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

      # Armazenar o resultado em uma lista.
      responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")

    # Fechar conexão com a IA (somente se você a usou ativamente)
    client_ai.close()

    # por motivo de performance as repostas foram adicionadas a uma lista.
    # No retorno a lista é convertida para um dataframe.
    return pd.DataFrame(responses)

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

Realizar consulta usando o modelo gpt-oss de 20 bilhões de parâmetros.

In [16]:
# Modelo gpt-oss de 20 bilhões de parâmetros.
model_id = 'openai/gpt-oss-20b'

# Dataframe com as respostas do primeiro modelo
df_gptoss_response = questions_submit(client_ai_instance('groq_api_key'), model_id, df_my_questions)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Modelo llama 4 scout de 17 bilhões de parâmetros.

In [12]:
# Modelo llama 4 scout, mais leve, de 17 bilhões de parâmetros.
model_id = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Dataframe com as respostas do primeiro modelo
df_llama4_response = questions_submit(client_ai_instance('groq_api_key'), model_id, df_my_questions)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Específico para o gemini

In [13]:
# Função para usar modelos de IA específica ao Gemini.
def questions_gemini_submit(client_ai, model_id, df_questions):
  # Criar uma lista vazia, para guardar as respostas, por questão de performance.
  gemini_responses = []

  # Repetição que percorre todo Dataframe.
  for index, row in df_my_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preparação do prompt em Markdown.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Chamada simples para a API da Google em nuvem.
      response = client_ai.models.generate_content(
          model=model_id,
          contents=prompt_usuario,
          config={
              "temperature": 0.1,  # Conservador.
              "max_output_tokens": 1024
          }
      )

      # Armazenar o resultado em uma lista.
      gemini_responses.append({"question": questao, "response": response.candidates[0].content.parts[0].text})
      print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA
  client_ai.close()

  # Para melhor visualização, converter as respostas em um Dataframe.
  return pd.DataFrame(gemini_responses)

Modelo gemini flash lançado em 2026. Gemini flash é o modelo mais leve quando comparado ao gemini pro.

In [14]:
# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview de 2026.
model_id = 'gemini-3.1-flash-lite-preview'

# Dataframe com as respostas do primeiro modelo
df_gemini_response = questions_gemini_submit(client_ai_instance('google_api_key'), model_id, df_my_questions)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Arrumar as respostas das 3 IAs.

In [42]:
# Renomear colunas dos dataframes para melhor legibilidade.
df_gptoss_response.rename(columns={'response': 'gptoss'}, inplace=True)
df_llama4_response.rename(columns={'response': 'llama4'}, inplace=True)
df_gemini_response.rename(columns={'response': 'gemini'}, inplace=True)
key = 'question'

# Realizar o primeiro inner join entre gptoss_responses e llama4_responses
df_gptoss_llama = merge_dataframes(df_gptoss_response, df_llama4_response, key)

# Realizar o segundo inner join com df_gemini_response
df_gptoss_llama_gemini = merge_dataframes(df_gptoss_llama, df_gemini_response, key)

# Junção com o Dataframe de respostas das IAs com o das perguntas, selecionando a coluna choices da linha de guia de respostas.
columns = ['question', 'statement', 'turns', 'gptoss', 'llama4', 'gemini', 'choices']
df_ias = merge_dataframes(df_gptoss_llama_gemini, df_my_questions, key, columns)

,question,statement,turns,gptoss,llama4,gemini,choices
0,31,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",,,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,"A medida judicial cabível é a ação anulatória,..."
1,32,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"A partir de quando se deve contar, no caso con...",**Prazo para a oferta dos embargos à execução ...,O prazo para oferta dos embargos à execução fi...,O prazo para a oferta dos Embargos à Execução ...,O prazo para oferta dos embargos à execução fi...
2,33,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,A qual dos municípios o ISS de jardinagem é de...,**ISS de jardinagem – Município de destino do ...,## Resposta\n\nO Imposto Sobre Serviços (ISS) ...,O Imposto Sobre Serviços de Qualquer Natureza ...,O ISS de jardinagem é devido ao Município Beta...
3,34,QUESTÃO\n\nA Administração Tributária Federal ...,É válida a exigência de depósito do montante c...,,A exigência de depósito do montante integral c...,"Não, a exigência de depósito do montante integ...","Não, pois é inconstitucional a exigência de de..."
4,35,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,Está correto o argumento da necessidade de not...,,## Resposta\n\n### Item 1: Correção do Argumen...,"Não, o argumento da sociedade empresária não e...","Não está correto, porque a entrega de declaraç..."
5,36,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,,,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...,EXCELENTÍSSIMO SENHOR DOUTOR DESEMBARGADOR PRE...,O(a) examinando(a) deve apresentar recurso de ...
6,37,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,Há necessidade de demonstração do elemento sub...,,A responsabilização administrativa da sociedad...,Para a caracterização da responsabilização adm...,Não. A responsabilização administrativa por at...
7,38,QUESTÃO\n\nDeterminada informação de interesse...,É lícita a cobrança efetuada pelo órgão respon...,**Resposta ao Questionamento**\n\n**I – Da leg...,A questão apresentada envolve a interpretação ...,"Não, a cobrança efetuada pelo órgão público é ...",Não. A submissão e o processamento de pedido d...
8,39,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,É possível a utilização do sistema de registro...,,"Sim, é possível a utilização do sistema de reg...","Sim, é possível a utilização do Sistema de Reg...",Sim. A Administração poderá contratar a execuç...
9,40,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",A aprovação de Iná no mencionado concurso impo...,,A aprovação de Iná no concurso público para a ...,Não. A aprovação de Iná no concurso público pa...,"Não. Iná foi aprovada para emprego público, ao..."


Métrica usada para avaliar as respostas, BERTScore.

In [40]:
# Função para calular a métrica, um valor entre 0 e 100%.
def calculate_bert(candidates, references):
  if not isinstance(candidates, list) or not isinstance(references, list):
    print("Candidates e References precisam ser lista.")
    return []
  if len(candidates) != len(references):
    print("Candidates e References precisam ter a mesma quantidade de elementos.")
    return []

  # Conferir se os valores são strings.
  processed_candidates = []
  for item in candidates:
    if isinstance(item, (str, int, float)):
      processed_candidates.append(str(item))
    else:
      processed_candidates.append("")

  processed_references = []
  for item in references:
    if isinstance(item, (str, int, float)):
      processed_references.append(str(item))
    else:
      processed_references.append("")

  # Depois das validações, realizao o calculo propriamente dito.
  try:
    P, R, F1 = score(processed_candidates, processed_references, lang="pt", verbose=False)
    return [f * 100 for f in F1.tolist()]
  except Exception as e:
    print(f"Erro ao tentar calcular BERTScore: {e}")
    return []

Calcular a métrica BERTStore para o Dataframe das respostas das IAs.

In [41]:
# The dataframe df_ias is already available from previous steps.
df_ias['gptoss_bert'] = calculate_bert(df_ias['gptoss'].tolist(), df_ias['choices'].tolist())
df_ias['llama4_bert'] = calculate_bert(df_ias['llama4'].tolist(), df_ias['choices'].tolist())
df_ias['gemini_bert'] = calculate_bert(df_ias['gemini'].tolist(), df_ias['choices'].tolist())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Erro ao tentar calcular BERTScore: BertTokenizer has no attribute build_inputs_with_special_tokens


ValueError: Length of values (0) does not match length of index (10)

Avaliar as respostas por um juiz on-line (IA).

In [ ]:
def llm_judge(client_ai, model_id, name_ia):
  # Criar lista vazia para guardar as respostas.
  responses = []

  # Repetição para percorrer todo Dataframe.
  for index, row in df_ias.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    pergunta = row['statement'] + ' ' + row['turns']
    baseline = row['choices']
    ia = row[name_ia]

    prompt_usuario = f"""
    Você é um jurista sênior especializado em Direito Administrativo e Tributário brasileiro.
    Sua função é atuar como juiz em um benchmark de IAs.
    Analise as respostas comparando-as rigorosamente com a Resposta Base (Gabarito).
    Atribua um percentual de acerto (0-100%). Traga apensa um número com duas casas decimais.
    PERGUNTA: {pergunta}

    RESPOSTA BASE (GABARITO): {baseline}
    ---
    RESPOSTA IA: {ia}
      ---
    [FORMATO DE RESPOSTA]
    Apenas um número com duas casas decimais.
    """
    # Realizar consulta a IA.
    completion = client_ai.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador
    )
    response = completion.choices[0].message.content

    # Armazenar o resultado em uma lista.
    responses.append({"question": questao, "response": response})
    print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA (somente se você a usou ativamente)
  client_ai.close()

  # por motivo de performance as repostas foram adicionadas a uma lista.
  # No retorno a lista é convertida para um dataframe.
  return pd.DataFrame(responses)

In [ ]:
model_id = "nvidia/nemotron-3-nano-30b-a3b:free"
df_score_lama3 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama3')
df_score_lama4 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama4')
df_score_gemini = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'gemini')

In [18]:
df_gptoss_response = df_llama_response.copy()